In [ ]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

# Feature panel — controlled, statistics-driven splits

This notebook partitions `DF_common_final_1` (protocol-level, **4 380 rows × 95
numeric features**, one year of 2-hour buckets) along the agreed *split umbrellas*.
Every split is **decision-based** — justified with statistics computed via
`adv_validation` and `EDA` — and shown **before vs after**, with a **balance check**
so no piece dominates by row count.

All split logic lives in **`feature_engineering.py`**; the statistical primitives live
in **`EDA.py`** / **`adv_validation.py`**. The notebook only *calls* and *proves*.

| # | Umbrella | Axis | Signal (split factor) | Pieces |
|---|----------|------|-----------------------|--------|
| 2 | Co-movement similarity | column | pairwise `\|corr\|` → clustering | redundancy groups |
| 3 | Tail-risk / volatility tier | column | excess-kurtosis · Hill α · robust-CV | stable / moderate / wild |
| 5 | Volatility regime | row | rolling σ of net liquidity flow | calm / normal / turbulent |
| 6 | Activity intensity | row | transaction **count** per bucket | quiet / active / peak |
| 7 | Whale dominance | row | avg transaction **size** (vol÷count) | retail / mixed / whale |

**Two axes, no overlap.** *Column* splits (2, 3) group the 95 features and keep all
rows. *Row* splits (5, 6, 7) cut the 4 380 rows into 3 balanced sub-frames, each on a
**distinct** signal — turbulence (5) ≠ how busy (6) ≠ how concentrated (7).

In [ ]:
# Libraries: pandas, numpy (framing); EDA, adv_validation (stats); feature_engineering (splits)
import pandas as pd
import numpy as np

import EDA
import adv_validation as adv
import feature_engineering as fe

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

df = pd.read_csv("transformed_data/DF_common_final_1.csv")
num = fe.numeric_columns(df)                       # 95 numeric features (time_bucket excluded)
print(f"frame: {df.shape[0]} rows x {df.shape[1]} cols  |  numeric features: {len(num)}")
df.head(3)

---
## Axis A — column splits (group the 95 features; all rows kept)

### Umbrella 2 — Co-movement similarity

**Idea.** Let the data group the columns: cluster features that move together so
redundant families are visible and one representative can stand in for many.
**Split factor.** Pairwise **signed** correlation `r` (range -1 to 1) over the 4 380-row
series → distance `1 - r` → agglomerative **complete-linkage** clustering (positively
co-moving columns group; anti-correlated ones stay apart; complete linkage avoids the
chaining that collapses everything into one mega-cluster).
**Decision basis (below).** Real redundancy exists — dozens of feature pairs are near-
perfectly correlated, so collapsing them is warranted, not arbitrary.

In [ ]:
# BEFORE — quantify redundancy across the 95 features (signed r in -1..1)
corr = df[num].corr()                                    # signed Pearson r
iu = np.triu_indices(len(num), k=1)
pairs = pd.Series(corr.to_numpy()[iu],
                  index=[f"{num[i]} ~ {num[j]}" for i, j in zip(*iu)])
print(f"{len(pairs)} feature pairs  |  |r|>0.90: {(pairs.abs()>0.90).sum()}"
      f"  |r|>0.95: {(pairs.abs()>0.95).sum()}  |r|>0.99: {(pairs.abs()>0.99).sum()}")
# strongest pairs by |r|, shown WITH sign (r in -1..1)
pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(8).round(3)

In [ ]:
# SPLIT — 6 co-movement clusters on signed r (-1..1), complete linkage (defaults)
res2 = fe.split_columns_by_correlation(df, n_clusters=6)
{name: len(cols) for name, cols in res2["groups"].items()}

In [ ]:
# AFTER (proof) — every cluster's internal mean r (signed, -1..1) sits well above the global average
fe.cluster_coherence(df, res2["groups"]).round(3)

In [ ]:
# the members of each co-movement family (first 8 shown)
for name, cols in res2["groups"].items():
    tail = " ..." if len(cols) > 8 else ""
    print(f"{name} ({len(cols)}): " + ", ".join(cols[:8]) + tail)

### Umbrella 3 — Tail-risk / volatility tier

**Idea.** Separate the wild, heavy-tailed columns from the calm, bounded ones so each
gets the right downstream treatment (tail-aware methods for the wild ones only).
**Split factor.** A composite per-column **`tail_score`** = mean of the percentile
ranks of excess kurtosis, robust CV (IQR), and Hill heaviness (`−hill α`). Columns are
cut into **equal-count terciles** → stable / moderate / wild (≈32 columns each).
**Decision basis (below).** The columns span an enormous tail range — most are fat-
tailed and many have infinite-variance tails — so a single treatment can't fit all.

In [ ]:
# BEFORE — how wide is the per-column tail behaviour? (justifies tiering)
scores = fe.column_tail_scores(df)
print(f"fat-tailed (excess_kurtosis > 3): {(scores['excess_kurtosis'] > 3).sum()} / {len(scores)}")
print(f"infinite-variance tail (Hill a < 2): {(scores['hill_tail_index'] < 2).sum()} / {len(scores)}")
scores[["excess_kurtosis", "hill_tail_index", "robust_cv_iqr", "tail_score"]].describe().round(2)

In [ ]:
# SPLIT — equal-count tercile tiers on the composite score
res3 = fe.split_columns_by_tail_risk(df)
{name: len(cols) for name, cols in res3["groups"].items()}

In [ ]:
# AFTER (proof) — metrics rise monotonically stable -> wild (kurtosis up, Hill a down, CV up)
fe.column_group_profile(df, res3["groups"]).round(2)

In [ ]:
for name, cols in res3["groups"].items():
    print(f"{name} ({len(cols)}): " + ", ".join(cols[:6]) + (" ..." if len(cols) > 6 else ""))

### Generic — binary column split by any `STAT_COLS` statistic

`fe.split_columns_by_stat(df, stat, threshold)` partitions the 95 features into **two**
frames by whether a single profile statistic — read from `adv_validation`'s
`statistical_validation` (computed there, not here) — is below or at/above a
**user-defined threshold**. The columns come from `df` itself; pass a precomputed
`stats` table to skip recomputation. Example: separate steady from volatile features by CV.

In [ ]:
# Two frames by coefficient of variation: low (cv < 1) vs high (cv >= 1).
# stats are computed in adv_validation (statistical_validation) if not passed; cols come from df.
low_cv, high_cv = fe.split_columns_by_stat(df, "cv", threshold=1.0)
print(f"cv < 1.0 : {low_cv.shape[1] - 1} cols   |   cv >= 1.0 : {high_cv.shape[1] - 1} cols")
high_cv.head(3)

---
## Axis B — row splits (cut the 4 380 rows into 3 balanced sub-frames)

Each split uses a **different** signal and produces equal-count terciles. Every
result is a dict: `score` (the signal), `labels` (calm/active/whale…), `frames`
(`{label: sub-DataFrame}`), `cutpoints`, `n_unlabeled`.

> **Caveat.** Row pieces are scattered in time (not contiguous), so the *time-ordered*
> `EDA` metrics — `autocorrelation`, `hurst_exponent`, the drawdown family — are **not**
> meaningful on a sub-frame. Distributional metrics (VaR, Gini, kurtosis, …) are fine.

### Umbrella 5 — Volatility regime (calm / normal / turbulent)

**Idea.** Segment time into market states by how turbulent the protocol is.
**Split factor.** Rolling σ (12 buckets = 1 day) of `net_liquidity_flow_usd`
(`EDA.rolling_volatility`) → terciles.
**Why not `market_stress_index_usd`?** It is ~72% **missing**, so its terciles would
cover <28% of rows — impossible to balance. It is *dense enough to validate against*,
though: where it is observed, the turbulent regime should carry more of it.

In [ ]:
# BEFORE — the sparse stress index can't be balanced; the rolling-vol signal is dense
stress = pd.to_numeric(df["market_stress_index_usd"], errors="coerce")
print(f"market_stress_index_usd: {stress.isna().mean()*100:.1f}% missing -> unusable as a balanced split")
signal = EDA.rolling_volatility(df, "net_liquidity_flow_usd", window=12)
print(f"rolling-vol signal: {signal.notna().sum()} / {len(df)} rows defined (dense)")
signal.describe().round(2)

In [ ]:
# SPLIT + balance
res5 = fe.split_by_volatility_regime(df)
print("signal:", res5["signal_name"], "| unlabeled (rolling head):", res5["n_unlabeled"])
fe.split_balance(res5["labels"])

In [ ]:
# AFTER (proof) — mean (tail-sensitive) stress & liquidations climb calm -> turbulent,
# validating the regime against the independent stress index it was NOT built from
fe.group_stat_table(res5["frames"],
    ["market_stress_index_usd", "liquidation_tx_count", "net_liquidity_flow_usd"],
    before=df, agg="mean").round(2)

### Umbrella 6 — Activity intensity (quiet / active / peak)

**Idea.** Segment time by *how busy* the protocol is — transaction **count**,
independent of dollar size (a quiet bucket can still hold one huge trade).
**Split factor.** `total_activity` per bucket → terciles.

In [ ]:
# SPLIT + balance
res6 = fe.split_by_activity_intensity(df)
print("signal:", res6["signal_name"])
fe.split_balance(res6["labels"])

In [ ]:
# AFTER (proof) — counts separate cleanly; user counts track the activity tiers
fe.group_stat_table(res6["frames"],
    ["total_activity", "supply_tx_count", "borrow_tx_count", "unique_borrowers"],
    before=df, agg="median").round(1)

### Umbrella 7 — Whale dominance (retail / mixed / whale)

**Idea.** Segment time by *how concentrated* volume is — average transaction **size**
(volume ÷ count). Whale buckets move large value in **few** transactions.
**Split factor.** `protocol_turnover_usd / total_activity` → terciles.
**Distinct from #6.** Same dollar volume can come from many small trades (retail) or a
few whales — so this is the count-normalised signal, not raw volume or raw count.

In [ ]:
# BEFORE — average transaction size is heavily whale-skewed
avg_tx = (pd.to_numeric(df["protocol_turnover_usd"], errors="coerce")
          / pd.to_numeric(df["total_activity"], errors="coerce"))
print(f"avg tx size: median ${avg_tx.median():,.0f}  max ${avg_tx.max():,.0f}"
      f"  ({avg_tx.max()/avg_tx.median():.0f}x median)")
print(f"top 1% of buckets hold {EDA.top_k_concentration(df, 'protocol_turnover_usd', k=0.01)*100:.1f}% of total turnover")
res7 = fe.split_by_whale_dominance(df)
print("signal:", res7["signal_name"])
fe.split_balance(res7["labels"])

In [ ]:
# AFTER (proof) — whale tier: highest turnover, FEWEST transactions (= few large trades)
fe.group_stat_table(res7["frames"],
    ["protocol_turnover_usd", "total_activity", "borrow_amount_value_usd"],
    before=df, agg="median").round(1)

In [ ]:
# AFTER (deeper, via adv_validation) — each tier is a more homogeneous sub-population:
# turnover skewness/kurtosis collapse from the full-panel "ALL" values within every tier
fe.robust_profile(res7["frames"], ["protocol_turnover_usd", "total_activity"],
                  before=df).round(3)

---
## Summary & how to use

All five splits are reproducible from `feature_engineering.py`:

```python
# column groups (dict {group: [columns]})
res2 = fe.split_columns_by_correlation(df, n_clusters=6)
res3 = fe.split_columns_by_tail_risk(df)

# row sub-frames (dict {label: DataFrame} under ["frames"])
res5 = fe.split_by_volatility_regime(df)      # calm / normal / turbulent
res6 = fe.split_by_activity_intensity(df)     # quiet / active / peak
res7 = fe.split_by_whale_dominance(df)        # retail / mixed / whale

turbulent = res5["frames"]["turbulent"]       # e.g. work on one regime
```

Proof helpers: `fe.split_balance`, `fe.group_stat_table`, `fe.cluster_coherence`,
`fe.column_group_profile`.

In [ ]:
# one-glance balance across all three row splits (requirement: ~even pieces)
summary = pd.DataFrame({
    "volatility_regime (5)": fe.split_balance(res5["labels"])["n"],
    "activity_intensity (6)": fe.split_balance(res6["labels"])["n"],
    "whale_dominance (7)":   fe.split_balance(res7["labels"])["n"],
})
print("row counts per tier (each ~1/3 of 4 380):")
summary